# TactiVision CV — Player Tracker (Colab GPU)

Run this notebook on Google Colab with a **T4 GPU** runtime.

**Setup:**
1. Upload `models/best.pt` and `data/match_5min.mp4` to your Google Drive
2. Put them in a folder called `TactiScout-CV/` on Drive
3. Change runtime to GPU: Runtime → Change runtime type → T4 GPU
4. Run all cells

In [ ]:
# Cell 1: Install dependencies
!pip install -q ultralytics supervision scikit-learn

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths - EDIT THESE if your folder structure is different
DRIVE_FOLDER = '/content/drive/MyDrive/TactiScout-CV'
MODEL_PATH = f'{DRIVE_FOLDER}/models/best.pt'
VIDEO_PATH = f'{DRIVE_FOLDER}/data/match_5min.mp4'
OUTPUT_FOLDER = f'{DRIVE_FOLDER}/output'

import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Verify files exist
assert os.path.exists(MODEL_PATH), f'Model not found at {MODEL_PATH}'
assert os.path.exists(VIDEO_PATH), f'Video not found at {VIDEO_PATH}'
print('✓ Model found')
print('✓ Video found')

In [ ]:
# Cell 3: Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 4: Write BoT-SORT config (tuned for football with Re-ID)
BOTSORT_CONFIG = '/content/botsort_football.yaml'

config_content = """# BoT-SORT for football - GPU config with Re-ID
tracker_type: botsort
track_high_thresh: 0.2
track_low_thresh: 0.05
new_track_thresh: 0.2
track_buffer: 150
match_thresh: 0.7
fuse_score: True

# Camera motion compensation
gmc_method: sparseOptFlow

# Re-ID ENABLED (fast on GPU)
proximity_thresh: 0.5
appearance_thresh: 0.3
with_reid: True
model: auto
"""

with open(BOTSORT_CONFIG, 'w') as f:
    f.write(config_content)

print('✓ BoT-SORT config written')

In [ ]:
# Cell 5: Run BoT-SORT Tracking (PASS 1)
import supervision as sv
from ultralytics import YOLO
import numpy as np
import cv2
import json
from collections import Counter, defaultdict
import time

# Class mapping
BALL_CLASS = 0
GOALKEEPER_CLASS = 1
PLAYER_CLASS = 2
REFEREE_CLASS = 3
TRACK_CLASSES = [PLAYER_CLASS, GOALKEEPER_CLASS]

# Load model
model = YOLO(MODEL_PATH)
print(f'✓ Model loaded: {MODEL_PATH}')

# Get video info
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
fps = video_info.fps or 25
print(f'  Video: {video_info.total_frames} frames, {fps} fps, {video_info.width}x{video_info.height}')

# --- TRACKING ---
print('\n=== Running BoT-SORT Tracking ===')
tracking_data = []
ball_positions = []
track_colors = defaultdict(list)

start_time = time.time()

results_gen = model.track(
    source=VIDEO_PATH,
    conf=0.15,
    iou=0.5,
    tracker=BOTSORT_CONFIG,
    stream=True,
    persist=True,
    verbose=False
)

for idx, results in enumerate(results_gen):
    frame = results.orig_img
    boxes = results.boxes

    if boxes is None or len(boxes) == 0:
        continue

    class_ids = boxes.cls.cpu().numpy().astype(int)
    confidences = boxes.conf.cpu().numpy()
    xyxy = boxes.xyxy.cpu().numpy()
    track_ids = boxes.id

    # --- Ball (no tracking needed) ---
    ball_mask = class_ids == BALL_CLASS
    if ball_mask.any():
        ball_boxes = xyxy[ball_mask]
        ball_confs = confidences[ball_mask]
        best = ball_confs.argmax()
        bx1, by1, bx2, by2 = ball_boxes[best]
        ball_positions.append({
            'frame': idx,
            'x': float((bx1 + bx2) / 2),
            'y': float((by1 + by2) / 2),
            'confidence': float(ball_confs[best])
        })

    # --- Players (tracked) ---
    if track_ids is None:
        continue

    track_ids_np = track_ids.cpu().numpy().astype(int)
    player_mask = np.isin(class_ids, TRACK_CLASSES)

    for i in np.where(player_mask)[0]:
        x1, y1, x2, y2 = xyxy[i]
        tid = int(track_ids_np[i])
        cls = int(class_ids[i])

        tracking_data.append({
            'frame': idx,
            'track_id': tid,
            'class_id': cls,
            'class_name': 'goalkeeper' if cls == GOALKEEPER_CLASS else 'player',
            'x': float((x1 + x2) / 2),
            'y': float((y1 + y2) / 2),
            'bbox': [float(x1), float(y1), float(x2), float(y2)],
            'confidence': float(confidences[i])
        })

        # Sample jersey color
        if idx % 5 == 0 and len(track_colors[tid]) < 20:
            ix1, iy1, ix2, iy2 = int(x1), int(y1), int(x2), int(y2)
            ix1, iy1 = max(0, ix1), max(0, iy1)
            ix2 = min(frame.shape[1], ix2)
            iy2 = min(frame.shape[0], iy2)
            crop = frame[iy1:iy2, ix1:ix2]
            if crop.size > 0 and crop.shape[0] >= 10 and crop.shape[1] >= 5:
                h, w = crop.shape[:2]
                torso = crop[int(h*0.15):int(h*0.45), :]
                if torso.size > 0:
                    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
                    color = np.median(hsv.reshape(-1, 3), axis=0)
                    track_colors[tid].append(color)

    if idx % 500 == 0:
        elapsed = time.time() - start_time
        fps_actual = (idx + 1) / elapsed if elapsed > 0 else 0
        unique_so_far = len(set(d['track_id'] for d in tracking_data))
        print(f'  Frame {idx}/{video_info.total_frames} | '
              f'Speed: {fps_actual:.1f} fps | Unique IDs: {unique_so_far}')

elapsed = time.time() - start_time
unique_raw = len(set(d['track_id'] for d in tracking_data))
ball_pct = len(ball_positions) / video_info.total_frames * 100

print(f'\n=== Pass 1 Done ===')
print(f'  Time: {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'  Raw unique IDs: {unique_raw}')
print(f'  Ball coverage: {len(ball_positions)}/{video_info.total_frames} ({ball_pct:.1f}%)')

In [ ]:
# Cell 6: PASS 2 - Merge fragments
print('=== PASS 2: Merging Fragments ===')

# Step A: Filter short tracks (< 1 second)
min_frames = int(fps)
id_counts = Counter(d['track_id'] for d in tracking_data)
valid_ids = {tid for tid, count in id_counts.items() if count >= min_frames}
tracking_data = [d for d in tracking_data if d['track_id'] in valid_ids]
print(f'  Filter (<{min_frames} frames): {len(id_counts)} → {len(valid_ids)} tracks')

# Step B: Merge by proximity
max_gap = int(fps * 4)
max_distance = 150

track_timelines = defaultdict(list)
for d in tracking_data:
    track_timelines[d['track_id']].append((d['frame'], d['x'], d['y']))
for tid in track_timelines:
    track_timelines[tid].sort()

track_info = {}
for tid, tl in track_timelines.items():
    track_info[tid] = {
        'start_frame': tl[0][0], 'end_frame': tl[-1][0],
        'start_pos': (tl[0][1], tl[0][2]),
        'end_pos': (tl[-1][1], tl[-1][2]),
        'length': len(tl)
    }

sorted_tracks = sorted(track_info.keys(), key=lambda t: -track_info[t]['length'])
merge_map = {}

for i, anchor in enumerate(sorted_tracks):
    if anchor in merge_map:
        continue
    anchor_end_frame = track_info[anchor]['end_frame']
    anchor_end_pos = track_info[anchor]['end_pos']

    for j in range(i + 1, len(sorted_tracks)):
        candidate = sorted_tracks[j]
        if candidate in merge_map:
            continue
        c_info = track_info[candidate]
        gap = c_info['start_frame'] - anchor_end_frame
        if gap < 1 or gap > max_gap:
            gap_rev = track_info[anchor]['start_frame'] - c_info['end_frame']
            if gap_rev < 1 or gap_rev > max_gap:
                continue
            dx = track_info[anchor]['start_pos'][0] - c_info['end_pos'][0]
            dy = track_info[anchor]['start_pos'][1] - c_info['end_pos'][1]
        else:
            dx = c_info['start_pos'][0] - anchor_end_pos[0]
            dy = c_info['start_pos'][1] - anchor_end_pos[1]

        dist = np.sqrt(dx**2 + dy**2)
        if dist < max_distance:
            if (c_info['start_frame'] > anchor_end_frame or
                    c_info['end_frame'] < track_info[anchor]['start_frame']):
                merge_map[candidate] = anchor
                if c_info['end_frame'] > anchor_end_frame:
                    anchor_end_frame = c_info['end_frame']
                    anchor_end_pos = c_info['end_pos']

tracking_data = [dict(d, track_id=merge_map.get(d['track_id'], d['track_id'])) for d in tracking_data]
print(f'  Proximity merge: {len(merge_map)} merged → {len(set(d["track_id"] for d in tracking_data))} IDs')

# Step C: Merge by appearance (jersey color)
active_ids = set(d['track_id'] for d in tracking_data)
track_avg_colors = {}
for tid, colors in track_colors.items():
    if tid in active_ids and len(colors) >= 3:
        track_avg_colors[tid] = np.median(colors, axis=0)

if len(track_avg_colors) >= 2:
    track_timelines2 = defaultdict(list)
    for d in tracking_data:
        track_timelines2[d['track_id']].append(d['frame'])
    track_spans = {tid: (min(f), max(f)) for tid, f in track_timelines2.items()}

    sorted_ids = sorted(track_avg_colors.keys(), key=lambda t: -(track_spans.get(t, (0,0))[1] - track_spans.get(t, (0,0))[0]))
    appearance_merge = {}
    color_threshold = 40.0
    max_gap_appearance = int(fps * 10)

    for i, anchor in enumerate(sorted_ids):
        if anchor in appearance_merge:
            continue
        anchor_color = track_avg_colors[anchor]
        anchor_start, anchor_end = track_spans.get(anchor, (0, 0))

        for j in range(i + 1, len(sorted_ids)):
            candidate = sorted_ids[j]
            if candidate in appearance_merge:
                continue
            cand_color = track_avg_colors[candidate]
            cand_start, cand_end = track_spans.get(candidate, (0, 0))

            if cand_start <= anchor_end and cand_end >= anchor_start:
                continue
            gap = min(abs(cand_start - anchor_end), abs(anchor_start - cand_end))
            if gap > max_gap_appearance:
                continue

            color_dist = np.linalg.norm(anchor_color - cand_color)
            if color_dist < color_threshold:
                appearance_merge[candidate] = anchor
                if cand_end > anchor_end:
                    anchor_end = cand_end
                    track_spans[anchor] = (track_spans[anchor][0], anchor_end)

    tracking_data = [dict(d, track_id=appearance_merge.get(d['track_id'], d['track_id'])) for d in tracking_data]
    print(f'  Appearance merge: {len(appearance_merge)} merged → {len(set(d["track_id"] for d in tracking_data))} IDs')

final_ids = len(set(d['track_id'] for d in tracking_data))
print(f'\n  FINAL: {final_ids} unique player IDs (target: ~22)')

In [ ]:
# Cell 7: Save results to Drive
players_path = f'{OUTPUT_FOLDER}/tracked_v5_players.json'
ball_path = f'{OUTPUT_FOLDER}/tracked_v5_ball.json'

with open(players_path, 'w') as f:
    json.dump(tracking_data, f)
with open(ball_path, 'w') as f:
    json.dump(ball_positions, f)

# Save track colors for later use
colors_path = f'{OUTPUT_FOLDER}/track_colors.json'
colors_serializable = {str(k): [c.tolist() for c in v] for k, v in track_colors.items()}
with open(colors_path, 'w') as f:
    json.dump(colors_serializable, f)

print(f'✓ Saved: {players_path}')
print(f'✓ Saved: {ball_path}')
print(f'✓ Saved: {colors_path}')
print(f'\nResults:')
print(f'  Unique player IDs: {final_ids}')
print(f'  Ball coverage: {len(ball_positions)}/{video_info.total_frames} ({ball_pct:.1f}%)')
print(f'  Total detections: {len(tracking_data)}')

In [ ]:
# Cell 8 (OPTIONAL): Write annotated output video
# This takes a few extra minutes but gives you a visual to review

WRITE_VIDEO = True  # Set to False to skip

if WRITE_VIDEO:
    print('Writing annotated video...')
    output_video = f'{OUTPUT_FOLDER}/tracked_v5.mp4'

    frame_players = defaultdict(list)
    for d in tracking_data:
        frame_players[d['frame']].append(d)
    frame_balls = {b['frame']: b for b in ball_positions}

    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.5)

    with sv.VideoSink(output_video, video_info) as sink:
        for idx, frame in enumerate(sv.get_video_frames_generator(VIDEO_PATH)):
            players = frame_players.get(idx, [])

            if players:
                xyxy = np.array([p['bbox'] for p in players])
                track_ids = np.array([p['track_id'] for p in players])
                confs = np.array([p.get('confidence', 0.5) for p in players])
                class_ids_arr = np.array([p.get('class_id', PLAYER_CLASS) for p in players])

                dets = sv.Detections(
                    xyxy=xyxy,
                    confidence=confs,
                    tracker_id=track_ids,
                    class_id=class_ids_arr
                )
                labels = [f'#{tid}' for tid in track_ids]
                frame = box_annotator.annotate(frame, dets)
                frame = label_annotator.annotate(frame, dets, labels)

            if idx in frame_balls:
                bx, by = int(frame_balls[idx]['x']), int(frame_balls[idx]['y'])
                cv2.circle(frame, (bx, by), 8, (0, 255, 255), -1)
                cv2.circle(frame, (bx, by), 10, (0, 0, 0), 2)

            sink.write_frame(frame)

            if idx % 1000 == 0:
                print(f'  Frame {idx}/{video_info.total_frames}')

    print(f'✓ Video saved: {output_video}')
else:
    print('Skipped video writing. Set WRITE_VIDEO = True to generate.')